## Audio RAG — Whisper (Full-Text) vs CLAP (Shared Semantic Space)

Experiments on audio input (MP3 from a YouTube video about Transformers / Attention).

### Pipeline comparison

| Aspect | Whisper pipeline | CLAP pipeline |
|---|---|---|
| Indexing | Audio → Whisper transcription → text embedding | Audio segments → CLAP audio encoder → audio embedding |
| Query encoding | Text embedding model | CLAP **text** encoder (same shared space) |
| Retrieval | Text ↔ Text similarity | Text ↔ Audio cross-modal similarity |
| Context for LLM | Transcribed text chunks | Whisper transcription of **retrieved** audio segments |
| BM25 | On transcription | Not applicable |
| Qdrant collection | `QDRANT_WHISPER_COLLECTION` | `QDRANT_CLAP_COLLECTION` |

> **Key insight**: in both pipelines the LLM receives text. What differs is *how* the relevant
> segments are found: keyword/semantic similarity on transcription (Whisper) vs
> cross-modal similarity in the CLIP-style audio-text space (CLAP).


### 1. Configuration

In [4]:
import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv("setup.env", override=True)

# ── Audio input ───────────────────────────────────────────────────────────────
# Path to the MP3 file. Accepted formats: mp3, wav, m4a, flac.
# Tip: download from YouTube with:
#   yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <URL>
AUDIO_PATH = os.getenv("AUDIO_PATH", "./content/audio.mp3")

# ── Whisper (Speech-to-Text) ──────────────────────────────────────────────────
#
# Backends:
#   "faster-whisper"  — recommended, fast GPU inference via CTranslate2
#   "hf"              — fallback using transformers.pipeline
#
# Good whisper model choices:
#   "deepdml/faster-whisper-large-v3-turbo-ct2" — fast + high quality
#   "Systran/faster-whisper-large-v3"           — better quality, slower
#   "Systran/faster-whisper-medium"             — smaller/faster fallback
WHISPER_BACKEND = os.getenv("WHISPER_BACKEND", "faster-whisper")
WHISPER_MODEL = os.getenv("WHISPER_MODEL", "deepdml/faster-whisper-large-v3-turbo-ct2")
WHISPER_LANGUAGE = os.getenv("WHISPER_LANGUAGE", "en")   # set to empty/None for auto-detect
WHISPER_BATCH_SIZE = int(os.getenv("WHISPER_BATCH_SIZE", "16"))  # RTX 4070: 16 is a good start
WHISPER_DEVICE = os.getenv("WHISPER_DEVICE", "cuda")         # "cuda" | "cpu" | "auto"
WHISPER_COMPUTE_TYPE = os.getenv("WHISPER_COMPUTE_TYPE", "float16") # use "int8_float16" if low VRAM
WHISPER_BEAM_SIZE = int(os.getenv("WHISPER_BEAM_SIZE", "1"))     # 1 = fastest; 5 = potentially better quality
WHISPER_VAD_FILTER = os.getenv("WHISPER_VAD_FILTER", "true").lower() == "true"

# ── Audio chunking ────────────────────────────────────────────────────────────
# Whisper returns segment-level timestamps. We merge consecutive segments
# into chunks of at most AUDIO_CHUNK_SECS seconds for indexing.
AUDIO_CHUNK_SECS  = int(os.getenv("AUDIO_CHUNK_SECS",   "45"))   # max seconds per text chunk
AUDIO_CHUNK_OVERLAP_SECS = int(os.getenv("AUDIO_CHUNK_OVERLAP_SECS", "5"))  # overlap between chunks

# CLAP splits the raw audio into fixed-length segments for embedding.
CLAP_SEGMENT_SECS = int(os.getenv("CLAP_SEGMENT_SECS",    "10"))  # seconds per CLAP segment
CLAP_SEGMENT_OVERLAP = int(os.getenv("CLAP_SEGMENT_OVERLAP",  "2"))  # overlap in seconds

# ── CLAP model ────────────────────────────────────────────────────────────────
# "laion/larger_clap_general"   — best general-purpose (music + speech + AudioSet)
# "laion/larger_clap_music"     — music-specialised
# "laion/clap-htsat-unfused"    — lighter baseline
CLAP_MODEL = os.getenv("CLAP_MODEL", "laion/larger_clap_general")
# CLAP_MODEL      = os.getenv("CLAP_MODEL", "laion/clap-htsat-unfused")

# ── Text embedding (Whisper pipeline) ─────────────────────────────────────────
# Same model as rag_pipeline_local.ipynb for fair comparison.
# "BAAI/bge-m3"                              — recommended (1024-dim)
# "sentence-transformers/all-MiniLM-L6-v2"  — lightweight (384-dim)
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── Retrieval ─────────────────────────────────────────────────────────────────
RETRIEVER_K = int(os.getenv("RETRIEVER_K",    "8"))
RERANKER_TOP_N = int(os.getenv("RERANKER_TOP_N", "4"))
RERANKER_MODEL = os.getenv("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
ENABLE_BM25 = os.getenv("ENABLE_BM25", "true").lower() == "true"
ENABLE_RERANKING  = os.getenv("ENABLE_RERANKING", "true").lower() == "true"

# ── Qdrant (local Docker) ─────────────────────────────────────────────────────
# docker run -d --name qdrant -p 6333:6333 -p 6334:6334 \
#   -v $(pwd)/qdrant_storage:/qdrant/storage qdrant/qdrant
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
QDRANT_WHISPER_COLLECTION = os.getenv("QDRANT_WHISPER_COLLECTION", "audio_whisper_v1")
QDRANT_CLAP_COLLECTION = os.getenv("QDRANT_CLAP_COLLECTION", "audio_clap_v1")
RESET_WHISPER_COLLECTION = os.getenv("RESET_WHISPER_COLLECTION", "true").lower() == "false"
RESET_CLAP_COLLECTION = os.getenv("RESET_CLAP_COLLECTION", "true").lower() == "false"

# ── Generation model ──────────────────────────────────────────────────────────
# Text-only generation (audio pipelines never pass raw audio to the LLM).
# "Qwen/Qwen2.5-3B-Instruct"          — local, CPU/GPU
# "Qwen/Qwen2.5-VL-3B-Instruct"       — also works (text-only prompt)
# Or use Ollama: set GENERATION_BACKEND=ollama and GENERATION_MODEL=mistral-nemo
GENERATION_MODEL = os.getenv("GENERATION_MODEL",   "Qwen/Qwen2.5-3B-Instruct")
GENERATION_BACKEND  = os.getenv("GENERATION_BACKEND", "hf")  # "hf" | "ollama"
GENERATION_MAX_NEW_TOKENS = int(os.getenv("GENERATION_MAX_NEW_TOKENS", "512"))
HF_DEVICE_MAP = os.getenv("HF_DEVICE_MAP",  "auto")
HF_TORCH_DTYPE = os.getenv("HF_TORCH_DTYPE", "auto")

# ── Persistence ───────────────────────────────────────────────────────────────
PERSIST_DIR = os.getenv("PERSIST_DIR", "./cache/audio/")
os.makedirs(PERSIST_DIR, exist_ok=True)
os.makedirs("./content/", exist_ok=True)

print("Configuration loaded.")
print(f"  Audio      : {AUDIO_PATH}")
print(f"  Whisper    : {WHISPER_MODEL} | backend={WHISPER_BACKEND} | lang={WHISPER_LANGUAGE or 'auto'}")
print(f"  Whisper HW : device={WHISPER_DEVICE}, compute={WHISPER_COMPUTE_TYPE}, batch={WHISPER_BATCH_SIZE}, beam={WHISPER_BEAM_SIZE}")
print(f"  Chunk      : {AUDIO_CHUNK_SECS}s (overlap={AUDIO_CHUNK_OVERLAP_SECS}s)")
print(f"  CLAP       : {CLAP_MODEL} | segment={CLAP_SEGMENT_SECS}s (overlap={CLAP_SEGMENT_OVERLAP}s)")
print(f"  Embeddings : {EMBEDDING_MODEL}")
print(f"  Retrieval  : k={RETRIEVER_K}, reranker_top_n={RERANKER_TOP_N}")
print(f"  Generator  : {GENERATION_MODEL} ({GENERATION_BACKEND})")
print(f"  Qdrant     : {QDRANT_URL}")
print(f"    Whisper collection : {QDRANT_WHISPER_COLLECTION}")
print(f"    CLAP collection    : {QDRANT_CLAP_COLLECTION}")


Configuration loaded.
  Audio      : ./content/audio.mp3
  Whisper    : deepdml/faster-whisper-large-v3-turbo-ct2 | backend=faster-whisper | lang=en
  Whisper HW : device=cuda, compute=float16, batch=16, beam=1
  Chunk      : 45s (overlap=5s)
  CLAP       : laion/larger_clap_general | segment=10s (overlap=2s)
  Embeddings : BAAI/bge-m3
  Retrieval  : k=10, reranker_top_n=5
  Generator  : mistral-nemo:latest (ollama)
  Qdrant     : http://localhost:6333
    Whisper collection : audio_whisper_v1
    CLAP collection    : audio_clap_v1


### 2. Audio Loading

Loads the MP3 and converts it to a 48 kHz mono waveform for CLAP audio embeddings.

Whisper transcription with the recommended `faster-whisper` backend reads `AUDIO_PATH`
directly, so it does not depend on this 48 kHz waveform. The Hugging Face fallback
resamples internally before ASR.

> If the audio file is not yet available, you can download it with:
> ```bash
> pip install yt-dlp
> yt-dlp -x --audio-format mp3 -o "content/audio.mp3" <YOUTUBE_URL>
> ```


In [5]:
import numpy as np
import librosa
from pathlib import Path

SAMPLE_RATE = 48000  # Hz — required by both Whisper and CLAP

def load_audio(path: str, sr: int = SAMPLE_RATE) -> np.ndarray:
    """
    Load an audio file and return a normalised float32 mono waveform at `sr` Hz.
    Supports mp3, wav, flac, m4a via librosa/ffmpeg.
    """
    audio_path = Path(path)
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Audio file not found: {path}\n"
            "Download with: yt-dlp -x --audio-format mp3 -o content/audio.mp3 <URL>"
        )
    waveform, _ = librosa.load(path, sr=sr, mono=True)
    print(f"Loaded: {audio_path.name}")
    print(f"  Duration : {len(waveform)/sr:.1f}s  ({len(waveform)/sr/60:.1f} min)")
    print(f"  Samples  : {len(waveform):,}  @ {sr} Hz")
    return waveform

waveform = load_audio(AUDIO_PATH)
AUDIO_DURATION_SECS = len(waveform) / SAMPLE_RATE


Loaded: audio.mp3
  Duration : 629.3s  (10.5 min)
  Samples  : 30,207,744  @ 48000 Hz


---
## 3. Whisper Pipeline — Full-Text Conversion

```
MP3 → Whisper → word-level segments + timestamps
    → merge into text chunks (≤ AUDIO_CHUNK_SECS)
    → text embedding (EMBEDDING_MODEL)
    → Qdrant QDRANT_WHISPER_COLLECTION
    → BM25 on transcription
    → hybrid retrieval → LLM (text only)
```


### 3.1 Transcription with Whisper

```bash
pip install faster-whisper
```

The notebook still supports the Hugging Face backend by setting:

```env
WHISPER_BACKEND=hf
WHISPER_MODEL=openai/whisper-large-v3
```


In [6]:
import torch

WHISPER_AVAILABLE = False
whisper_pipe = None
whisper_fw_model = None
whisper_fw_batched_model = None

def _resolve_whisper_device() -> str:
    """Resolve requested Whisper device, with a safe CPU fallback."""
    requested = (WHISPER_DEVICE or "auto").lower()
    if requested == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        print("  [warn] WHISPER_DEVICE=cuda requested, but CUDA is not available. Falling back to CPU.")
        return "cpu"
    return requested

whisper_device = _resolve_whisper_device()
print(f"Loading Whisper model: {WHISPER_MODEL} ({WHISPER_BACKEND}) ...")

if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
    try:
        from faster_whisper import WhisperModel, BatchedInferencePipeline

        whisper_fw_model = WhisperModel(
            WHISPER_MODEL,
            device=whisper_device,
            compute_type=WHISPER_COMPUTE_TYPE,
        )
        whisper_fw_batched_model = BatchedInferencePipeline(model=whisper_fw_model)
        WHISPER_AVAILABLE = True
        print(
            f"  ✓ faster-whisper ready on {whisper_device} "
            f"(compute={WHISPER_COMPUTE_TYPE}, batch={WHISPER_BATCH_SIZE})"
        )
    except Exception as e:
        print(f"  ✗ Failed to load faster-whisper: {e}")
        print("    Install with: pip install faster-whisper")
        WHISPER_AVAILABLE = False

elif WHISPER_BACKEND.lower() in {"hf", "transformers", "huggingface"}:
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline as hf_pipeline

    torch_dt = torch.float16 if whisper_device == "cuda" else torch.float32
    try:
        whisper_processor = AutoProcessor.from_pretrained(WHISPER_MODEL)
        whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
            WHISPER_MODEL,
            torch_dtype=torch_dt,
            low_cpu_mem_usage=True,
        ).to(whisper_device)

        whisper_pipe = hf_pipeline(
            "automatic-speech-recognition",
            model=whisper_model,
            tokenizer=whisper_processor.tokenizer,
            feature_extractor=whisper_processor.feature_extractor,
            torch_dtype=torch_dt,
            device=0 if whisper_device == "cuda" else -1,
            return_timestamps=True,
            chunk_length_s=30,
            batch_size=WHISPER_BATCH_SIZE,
            generate_kwargs={"language": WHISPER_LANGUAGE} if WHISPER_LANGUAGE else {},
        )
        WHISPER_AVAILABLE = True
        print(f"  ✓ Hugging Face Whisper ready on {whisper_device}")
    except Exception as e:
        print(f"  ✗ Failed to load Hugging Face Whisper: {e}")
        WHISPER_AVAILABLE = False

else:
    raise ValueError(
        "Unsupported WHISPER_BACKEND. Use 'faster-whisper' or 'hf'. "
        f"Got: {WHISPER_BACKEND!r}"
    )


Loading Whisper model: deepdml/faster-whisper-large-v3-turbo-ct2 (faster-whisper) ...
  ✓ faster-whisper ready on cuda (compute=float16, batch=16)


In [7]:
import json, hashlib, re
from pathlib import Path

# ── Transcription cache ───────────────────────────────────────────────────────
# Whisper on a long audio file is slow — cache to disk to avoid re-running.
def _audio_hash(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()[:12]

def _safe_cache_part(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-")

_cache_name = "_".join([
    "transcript",
    _audio_hash(AUDIO_PATH),
    _safe_cache_part(WHISPER_BACKEND),
    _safe_cache_part(WHISPER_MODEL.split("/")[-1]),
    _safe_cache_part(WHISPER_COMPUTE_TYPE),
]) + ".json"
TRANSCRIPT_CACHE = Path(PERSIST_DIR) / _cache_name


def _transcribe_faster_whisper(audio_path: str) -> dict:
    """Transcribe AUDIO_PATH with faster-whisper and return HF-like {text, chunks}."""
    language = WHISPER_LANGUAGE or None
    segments_iter, info = whisper_fw_batched_model.transcribe(
        audio_path,
        batch_size=WHISPER_BATCH_SIZE,
        language=language,
        vad_filter=WHISPER_VAD_FILTER,
        beam_size=WHISPER_BEAM_SIZE,
        word_timestamps=False,
    )

    chunks = []
    texts = []
    for seg in segments_iter:
        text = seg.text.strip()
        if not text:
            continue
        chunks.append({"text": text, "timestamp": [float(seg.start), float(seg.end)]})
        texts.append(text)

    detected_language = getattr(info, "language", None)
    language_probability = getattr(info, "language_probability", None)
    if detected_language:
        print(f"  Detected language: {detected_language} ({language_probability:.2f})")

    return {
        "text": " ".join(texts).strip(),
        "chunks": chunks,
        "metadata": {
            "backend": "faster-whisper",
            "model": WHISPER_MODEL,
            "device": whisper_device,
            "compute_type": WHISPER_COMPUTE_TYPE,
            "batch_size": WHISPER_BATCH_SIZE,
            "beam_size": WHISPER_BEAM_SIZE,
            "vad_filter": WHISPER_VAD_FILTER,
            "detected_language": detected_language,
            "language_probability": language_probability,
        },
    }


def _transcribe_hf(waveform: np.ndarray) -> dict:
    """Transcribe waveform with the Hugging Face ASR pipeline."""
    result = whisper_pipe(waveform.copy(), return_timestamps=True)
    return {
        "text": result["text"],
        "chunks": [
            {"text": c["text"].strip(), "timestamp": list(c["timestamp"])}
            for c in result.get("chunks", [])
            if c.get("text", "").strip()
        ],
        "metadata": {
            "backend": "hf",
            "model": WHISPER_MODEL,
            "device": whisper_device,
        },
    }


def transcribe(waveform: np.ndarray, cache_path: Path, audio_path: str = AUDIO_PATH) -> dict:
    """Transcribe audio and return a common dict with 'text' and timestamped 'chunks'."""
    if cache_path.exists():
        print(f"Loading transcription from cache: {cache_path.name}")
        with open(cache_path, encoding="utf-8") as f:
            return json.load(f)

    if not WHISPER_AVAILABLE:
        raise RuntimeError("Whisper model not loaded.")

    print("Transcribing audio...")
    if WHISPER_BACKEND.lower() in {"faster-whisper", "faster_whisper", "ct2"}:
        serialisable = _transcribe_faster_whisper(audio_path)
    else:
        serialisable = _transcribe_hf(waveform)

    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(serialisable, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Transcription saved to cache: {cache_path.name}")
    return serialisable


transcription = transcribe(waveform, TRANSCRIPT_CACHE, AUDIO_PATH)
full_text = transcription["text"]
segments  = transcription["chunks"]   # list of {text, timestamp: [start, end]}

print(f"\nTranscription complete: {len(segments)} segments, {len(full_text)} chars")
print("\nFirst 500 chars:")
print(full_text[:500])


Loading transcription from cache: transcript_2bad6f742512_faster-whisper_faster-whisper-large-v3-turbo-ct2_float16.json

Transcription complete: 23 segments, 9614 chars

First 500 chars:
BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models. BERT is designed to pre-trained deep bi-directional representations from unlabeled text by jointly conditioning on both left and right context in


### 3.2 Text Chunking with Timestamps

In [8]:
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class AudioChunk:
    """
    A chunk of transcribed audio with timestamp metadata.
    doc_type is always 'text' for both pipelines — the LLM only receives text.
    """
    content:    str            # transcribed text of this chunk
    doc_type:   str = "text"
    start_sec:  float = 0.0   # start time in the original audio
    end_sec:    float = 0.0   # end time in the original audio
    source_file: str = ""
    metadata:   dict = field(default_factory=dict)

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"


def merge_segments_into_chunks(
    segments:     List[dict],
    max_secs:     int = AUDIO_CHUNK_SECS,
    overlap_secs: int = AUDIO_CHUNK_OVERLAP_SECS,
    source_file:  str = AUDIO_PATH,
) -> List[AudioChunk]:
    """
    Merge consecutive Whisper segments into chunks of at most `max_secs` seconds.
    Adds `overlap_secs` of context from the previous chunk to each new chunk.

    Each Whisper segment has: {"text": str, "timestamp": [start, end]}
    End may be None for the last segment — use audio duration as fallback.
    """
    if not segments:
        return []

    chunks:  List[AudioChunk] = []
    buf_text:  List[str]   = []
    buf_segs:  List[dict]  = []
    buf_start: float       = segments[0]["timestamp"][0] or 0.0

    def flush(buf_text, buf_segs, buf_start):
        if not buf_text:
            return
        text  = " ".join(buf_text).strip()
        end   = buf_segs[-1]["timestamp"][1] or AUDIO_DURATION_SECS
        chunks.append(AudioChunk(
            content    = text,
            start_sec  = buf_start,
            end_sec    = end,
            source_file= source_file,
        ))

    overlap_buf: List[dict] = []   # segments carried over for context

    for seg in segments:
        ts    = seg["timestamp"]
        start = ts[0] if ts[0] is not None else (buf_start if buf_segs else 0.0)
        end   = ts[1] if ts[1] is not None else AUDIO_DURATION_SECS

        # Start a new chunk if max duration is exceeded
        if buf_segs and (end - buf_start) > max_secs:
            flush(buf_text, buf_segs, buf_start)
            # Carry-over overlap segments
            overlap_buf = [s for s in buf_segs if (s["timestamp"][1] or AUDIO_DURATION_SECS) >= (buf_start + max_secs - overlap_secs)]
            buf_text  = [s["text"] for s in overlap_buf]
            buf_segs  = list(overlap_buf)
            buf_start = overlap_buf[0]["timestamp"][0] if overlap_buf else start

        buf_text.append(seg["text"])
        buf_segs.append(seg)

    flush(buf_text, buf_segs, buf_start)
    return chunks


whisper_chunks = merge_segments_into_chunks(segments)
print(f"Created {len(whisper_chunks)} text chunks from {len(segments)} Whisper segments.")
print(f"  avg duration : {sum(c.end_sec - c.start_sec for c in whisper_chunks)/len(whisper_chunks):.1f}s")
print(f"\nFirst chunk {whisper_chunks[0].timestamp_label}:")
print(whisper_chunks[0].content[:300])


Created 23 text chunks from 23 Whisper segments.
  avg duration : 27.4s

First chunk [00:00 – 00:25]:
BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Trans


### 3.3 Text Embedding + Qdrant

In [9]:
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

# ── Text embedding model ──────────────────────────────────────────────────────
print(f"Loading text embedding model: {EMBEDDING_MODEL} ...")
text_embedding = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {"trust_remote_code": True},
    encode_kwargs = {"normalize_embeddings": True},
)
EMBEDDING_DIM = len(text_embedding.embed_query("dim probe"))
print(f"  ✓ Text embedding ready (dim={EMBEDDING_DIM})")

# ── Qdrant client (reusable across both pipelines) ────────────────────────────
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)

# ── Whisper collection ────────────────────────────────────────────────────────
if RESET_WHISPER_COLLECTION:
    client.delete_collection(QDRANT_WHISPER_COLLECTION)
    print(f"✓ Deleted '{QDRANT_WHISPER_COLLECTION}'")

if not client.collection_exists(QDRANT_WHISPER_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_WHISPER_COLLECTION,
        vectors_config  = VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created Whisper collection '{QDRANT_WHISPER_COLLECTION}' (dim={EMBEDDING_DIM})")
else:
    print(f"✓ Using existing Whisper collection '{QDRANT_WHISPER_COLLECTION}'")

whisper_vector_store = QdrantVectorStore(
    client          = client,
    collection_name = QDRANT_WHISPER_COLLECTION,
    embedding       = text_embedding,
)

# ── Docstore: UUID → AudioChunk ──────────────────────────────────────────────
whisper_docstore: Dict[str, AudioChunk] = {}


Loading text embedding model: BAAI/bge-m3 ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 43363.73it/s]


  ✓ Text embedding ready (dim=1024)
✓ Using existing Whisper collection 'audio_whisper_v1'


In [10]:
from langchain_core.documents import Document

def index_whisper_chunks(
    chunks:       List[AudioChunk],
    vector_store,
    docstore:     dict,
    batch_size:   int = 64,
) -> List[str]:
    """Index transcribed text chunks into Qdrant Whisper collection."""
    all_ids: List[str] = []
    lc_docs: List[Document] = []

    for chunk in chunks:
        if not chunk.content.strip():
            continue
        uid = str(uuid.uuid4())
        all_ids.append(uid)
        docstore[uid] = chunk
        lc_docs.append(Document(
            page_content = chunk.content,
            metadata     = {
                "doc_id":    uid,
                "doc_type":  "text",
                "start_sec": chunk.start_sec,
                "end_sec":   chunk.end_sec,
                "timestamp": chunk.timestamp_label,
                "source":    chunk.source_file,
            },
        ))

    print(f"Indexing {len(lc_docs)} chunks into '{QDRANT_WHISPER_COLLECTION}' ...")
    for i in range(0, len(lc_docs), batch_size):
        batch = lc_docs[i : i + batch_size]
        try:
            vector_store.add_documents(batch)
        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e}")
        print(f"  [{min(i+batch_size, len(lc_docs))}/{len(lc_docs)}] inserted")

    print(f"✓ Whisper indexing complete: {len(all_ids)} chunks.")
    return all_ids

whisper_ids = index_whisper_chunks(whisper_chunks, whisper_vector_store, whisper_docstore)


Indexing 23 chunks into 'audio_whisper_v1' ...
  [23/23] inserted
✓ Whisper indexing complete: 23 chunks.


### 3.4 BM25 on Transcription

In [11]:
import numpy as np
from rank_bm25 import BM25Okapi
from typing import Tuple

class AudioBM25Index:
    """BM25 index over AudioChunk content."""
    def __init__(self):
        self._chunks: List[AudioChunk] = []
        self._bm25                     = None

    def build(self, chunks: List[AudioChunk]) -> None:
        self._chunks  = chunks
        tokenized     = [c.content.lower().split() for c in chunks]
        self._bm25    = BM25Okapi(tokenized)
        print(f"BM25 built on {len(chunks)} Whisper chunks.")

    def retrieve(self, query: str, top_k: int = RETRIEVER_K) -> List[Tuple[float, AudioChunk]]:
        if self._bm25 is None or not self._chunks:
            return []
        scores  = self._bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self._chunks[i]) for i in top_idx if scores[i] > 0]

whisper_bm25 = AudioBM25Index()
if ENABLE_BM25:
    whisper_bm25.build(whisper_chunks)
else:
    print("BM25 disabled.")


BM25 built on 23 Whisper chunks.


### 3.5 Whisper Hybrid Retrieval

In [12]:
from sentence_transformers import CrossEncoder

# ── Cross-encoder reranker ─────────────────────────────────────────────────
if ENABLE_RERANKING:
    print(f"Loading cross-encoder: {RERANKER_MODEL} ...")
    try:
        reranker           = CrossEncoder(RERANKER_MODEL)
        RERANKER_AVAILABLE = True
        print("  ✓ Reranker ready.")
    except Exception as e:
        print(f"  ✗ {e}")
        reranker           = None
        RERANKER_AVAILABLE = False
else:
    reranker           = None
    RERANKER_AVAILABLE = False
    print("Reranking disabled.")


def whisper_hybrid_retrieve(query: str) -> List[AudioChunk]:
    """
    Hybrid retrieval on the Whisper pipeline:
      1. Dense: text query → text embedding → Qdrant Whisper collection
      2. BM25 : keyword matching on raw transcription chunks
      3. Deduplication by content
      4. Cross-encoder reranking on merged candidates
    """
    seen:       set                     = set()
    candidates: List[Tuple[str, AudioChunk]] = []   # (snippet, chunk)

    # Dense
    try:
        dense_hits = whisper_vector_store.similarity_search(query, k=RETRIEVER_K)
        for lc in dense_hits:
            uid   = lc.metadata.get("doc_id")
            chunk = whisper_docstore.get(uid) if uid else None
            if chunk is None:
                chunk = AudioChunk(content=lc.page_content,
                                   start_sec=lc.metadata.get("start_sec", 0),
                                   end_sec=lc.metadata.get("end_sec", 0))
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((lc.page_content, chunk))
    except Exception as e:
        print(f"[whisper_retrieve] Dense error: {e}")

    # BM25
    if ENABLE_BM25:
        for score, chunk in whisper_bm25.retrieve(query, top_k=RETRIEVER_K):
            if chunk.content not in seen:
                seen.add(chunk.content)
                candidates.append((chunk.content[:500], chunk))

    if not candidates:
        return []

    # Reranking
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, s) for s, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [c for _, c in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [c for _, c in ranked[:RERANKER_TOP_N]]
    else:
        final  = [c for _, c in candidates[:RERANKER_TOP_N]]

    print(f"[whisper_retrieve] {len(candidates)} candidates → {len(final)} "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_w = whisper_hybrid_retrieve("What is the attention mechanism?")
print(f"\nRetrieved {len(test_w)} chunks:")
for i, c in enumerate(test_w):
    print(f"  [{i}] {c.timestamp_label}  {c.content[:80]!r}")


Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2 ...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3589.26it/s]


  ✓ Reranker ready.
[whisper_retrieve] 11 candidates → 5 (query: 'What is the attention mechanism?')

Retrieved 5 chunks:
  [0] [09:10 – 09:37]  'For the pre-training corpus, we use the books corpus and English Wikipedia, extr'
  [1] [02:37 – 03:05]  'The authors use a left-to-right architecture, where every token can only attend '
  [2] [01:47 – 02:14]  'uses task-specific architectures that include the pre-trained representations as'
  [3] [10:03 – 10:29]  'Ablation studies confirm that the deep bidirectional architecture, enabled by th'
  [4] [01:21 – 01:47]  'which aim to predict the relationships between sentences by analyzing them holis'


---
## 4. CLAP Pipeline — Shared Audio-Text Semantic Space

```
MP3 → split into fixed-length segments (CLAP_SEGMENT_SECS)
    → CLAP audio encoder → audio embeddings
    → Qdrant QDRANT_CLAP_COLLECTION

Query (text) → CLAP text encoder → similarity search → top-k audio segments
    → resolve segment timestamps → extract matching Whisper transcription
    → LLM (text only)
```

The CLAP text encoder and audio encoder share the same vector space,
enabling cross-modal retrieval: a text query finds audio segments by
semantic similarity without requiring keyword overlap in the transcription.


### 4.1 CLAP Model

In [13]:
from transformers import ClapModel, ClapProcessor
import torch
import numpy as np

print(f"Loading CLAP model: {CLAP_MODEL} ...")
clap_device = "cuda" if torch.cuda.is_available() else "cpu"

def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out

    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output

    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state.mean(dim=1)

    if isinstance(out, (tuple, list)):
        return out[0]

    raise TypeError(f"Unsupported CLAP output type: {type(out)}")

try:
    clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL)

    clap_model = ClapModel.from_pretrained(
        CLAP_MODEL,
        torch_dtype=torch.float32,   # <- cambia qui
    ).to(clap_device)

    clap_model.eval()

    with torch.no_grad():
        dummy_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)
        dummy_inputs = clap_processor(
            audio=dummy_audio,
            return_tensors="pt",
            sampling_rate=SAMPLE_RATE,
        )

        dummy_inputs = {k: v.to(clap_device) for k, v in dummy_inputs.items()}

        out = clap_model.get_audio_features(**dummy_inputs)

        audio_emb = clap_output_to_tensor(out)
        audio_emb = torch.nn.functional.normalize(audio_emb, dim=-1)

        CLAP_DIM = audio_emb.shape[-1]

    CLAP_AVAILABLE = True
    print(f"  ✓ CLAP ready on {clap_device} | dim={CLAP_DIM}")

except Exception as e:
    print(f"  ✗ Could not load CLAP: {e}")
    clap_model = clap_processor = None
    CLAP_AVAILABLE = False
    CLAP_DIM = 512

Loading CLAP model: laion/larger_clap_general ...


Loading weights: 100%|██████████| 555/555 [00:00<00:00, 46063.89it/s]


  ✓ CLAP ready on cuda | dim=512


### 4.2 Audio Segmentation and CLAP Embedding

In [14]:
@dataclass
class AudioSegment:
    """A fixed-length audio segment with timestamp and precomputed CLAP embedding."""
    start_sec:   float
    end_sec:     float
    source_file: str
    transcript:  str = ""    # Whisper text for this time range (populated later)
    doc_type:    str = "text"  # always "text" — LLM receives transcript, not audio

    @property
    def timestamp_label(self) -> str:
        def fmt(s):
            m, sec = divmod(int(s), 60)
            return f"{m:02d}:{sec:02d}"
        return f"[{fmt(self.start_sec)} – {fmt(self.end_sec)}]"

    @property
    def content(self) -> str:
        """Alias for compatibility with metric functions that expect .content."""
        return self.transcript

def clap_output_to_tensor(out):
    if isinstance(out, torch.Tensor):
        return out
    if hasattr(out, "pooler_output"):
        return out.pooler_output
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state.mean(dim=1)
    raise TypeError(f"Unexpected CLAP output type: {type(out)}")

def split_waveform_into_segments(
    waveform:     np.ndarray,
    sr:           int   = SAMPLE_RATE,
    seg_secs:     int   = CLAP_SEGMENT_SECS,
    overlap_secs: int   = CLAP_SEGMENT_OVERLAP,
    source_file:  str   = AUDIO_PATH,
) -> List[AudioSegment]:
    """Split a waveform into overlapping fixed-length AudioSegments."""
    step   = (seg_secs - overlap_secs) * sr
    length = seg_secs * sr
    segs   = []
    pos    = 0
    while pos < len(waveform):
        end_sample = min(pos + length, len(waveform))
        start_s    = pos / sr
        end_s      = end_sample / sr
        segs.append(AudioSegment(start_sec=start_s, end_sec=end_s, source_file=source_file))
        if end_sample == len(waveform):
            break
        pos += step
    return segs


def embed_audio_segments_clap(
    waveform:  np.ndarray,
    segments:  List[AudioSegment],
    batch_size: int = 8,
) -> np.ndarray:
    """
    Embed each AudioSegment with the CLAP audio encoder.
    Returns (N, CLAP_DIM) float32 array, L2-normalised.
    """
    if not CLAP_AVAILABLE:
        return np.zeros((len(segments), CLAP_DIM), dtype=np.float32)

    all_vecs = []
    total    = len(segments)
    print(f"Embedding {total} audio segments with CLAP ...")

    for i in range(0, total, batch_size):
        batch_segs = segments[i : i + batch_size]
        batch_audio = []
        for seg in batch_segs:
            s = int(seg.start_sec * SAMPLE_RATE)
            e = int(seg.end_sec   * SAMPLE_RATE)
            batch_audio.append(waveform[s:e].astype(np.float32))

        try:
            inputs = clap_processor(
                audio=batch_audio,
                return_tensors="pt",
                sampling_rate=SAMPLE_RATE,
                padding=True,
            )

            inputs = {k: v.to(clap_device) for k, v in inputs.items()}

            with torch.no_grad():
                out = clap_model.get_audio_features(**inputs)
                vecs = clap_output_to_tensor(out)
                vecs = torch.nn.functional.normalize(vecs, dim=-1).float()

            all_vecs.append(vecs.cpu().numpy())

        except Exception as e:
            print(f"  ✗ Batch {i//batch_size}: {e} — using zero vectors")
            all_vecs.append(np.zeros((len(batch_segs), CLAP_DIM), dtype=np.float32))

        print(f"  [{min(i+batch_size, total)}/{total}] embedded")

    return np.vstack(all_vecs)


# Segment + embed
clap_segments = split_waveform_into_segments(waveform)
clap_vectors  = embed_audio_segments_clap(waveform, clap_segments)

print(f"\nSegments : {len(clap_segments)}")
print(f"Avg dur  : {sum(s.end_sec-s.start_sec for s in clap_segments)/len(clap_segments):.1f}s")
print(f"Vectors  : {clap_vectors.shape}")


Embedding 79 audio segments with CLAP ...
  [8/79] embedded
  [16/79] embedded
  [24/79] embedded
  [32/79] embedded
  [40/79] embedded
  [48/79] embedded
  [56/79] embedded
  [64/79] embedded
  [72/79] embedded
  [79/79] embedded

Segments : 79
Avg dur  : 9.9s
Vectors  : (79, 512)


### 4.3 Attach Whisper Transcription to CLAP Segments

In [15]:
def get_transcript_for_range(
    segments: List[dict],
    start_s:  float,
    end_s:    float,
    padding:  float = 1.0,
) -> str:
    """
    Extract the Whisper transcription covering [start_s - padding, end_s + padding].
    Uses the segment-level timestamps already available from the transcription.
    """
    relevant = []
    for seg in segments:
        ts_start = seg["timestamp"][0] or 0.0
        ts_end   = seg["timestamp"][1] or AUDIO_DURATION_SECS
        # Include segment if it overlaps with the query window
        if ts_end >= (start_s - padding) and ts_start <= (end_s + padding):
            relevant.append(seg["text"])
    return " ".join(relevant).strip()


# Attach transcript to every CLAP segment
print("Attaching Whisper transcription to CLAP segments ...")
for seg in clap_segments:
    seg.transcript = get_transcript_for_range(segments, seg.start_sec, seg.end_sec)

# How many segments have non-empty transcription?
with_text = sum(1 for s in clap_segments if s.transcript.strip())
print(f"✓ {with_text}/{len(clap_segments)} segments have transcription coverage.")

# Preview
print(f"\nFirst CLAP segment {clap_segments[0].timestamp_label}:")
print(f"  Transcript: {clap_segments[0].transcript[:200]!r}")


Attaching Whisper transcription to CLAP segments ...
✓ 79/79 segments have transcription coverage.

First CLAP segment [00:00 – 00:10]:
  Transcript: 'BERT, Pre-Training of Deep Bidirectional Transformers for Language Understanding. Jacob Devlin-Ming, Wei-Chong Kinton, Lee-Christina Tutanova. Google AI Language. Abstract. We introduce a new language'


### 4.4 CLAP Qdrant Collection + Indexing

In [16]:
from qdrant_client.http.models import PointStruct

# ── CLAP collection ───────────────────────────────────────────────────────────
if RESET_CLAP_COLLECTION and client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.delete_collection(QDRANT_CLAP_COLLECTION)
    print(f"✓ Deleted '{QDRANT_CLAP_COLLECTION}'")

if not client.collection_exists(QDRANT_CLAP_COLLECTION):
    client.create_collection(
        collection_name = QDRANT_CLAP_COLLECTION,
        vectors_config  = VectorParams(size=CLAP_DIM, distance=Distance.COSINE),
    )
    print(f"✓ Created CLAP collection '{QDRANT_CLAP_COLLECTION}' (dim={CLAP_DIM})")
else:
    print(f"✓ Using existing CLAP collection '{QDRANT_CLAP_COLLECTION}'")

# Docstore: UUID → AudioSegment
clap_docstore: Dict[str, AudioSegment] = {}

# ── Index all segments ────────────────────────────────────────────────────────
UPSERT_BATCH = 64
clap_ids     = []
points       = []

for seg, vec in zip(clap_segments, clap_vectors):
    uid = str(uuid.uuid4())
    clap_ids.append(uid)
    clap_docstore[uid] = seg
    points.append(PointStruct(
        id      = uid,
        vector  = vec.tolist(),
        payload = {
            "start_sec": seg.start_sec,
            "end_sec":   seg.end_sec,
            "timestamp": seg.timestamp_label,
            "source":    seg.source_file,
            "preview":   seg.transcript[:100],
        },
    ))

print(f"Inserting {len(points)} CLAP vectors into '{QDRANT_CLAP_COLLECTION}' ...")
inserted = 0
for i in range(0, len(points), UPSERT_BATCH):
    batch = points[i : i + UPSERT_BATCH]
    try:
        client.upsert(collection_name=QDRANT_CLAP_COLLECTION, points=batch)
        inserted += len(batch)
    except Exception as e:
        print(f"  ✗ Batch {i//UPSERT_BATCH}: {e}")

print(f"✓ CLAP indexing complete: {inserted}/{len(points)} segments.")


✓ Using existing CLAP collection 'audio_clap_v1'
Inserting 79 CLAP vectors into 'audio_clap_v1' ...
✓ CLAP indexing complete: 79/79 segments.


### 4.5 CLAP Hybrid Retrieval

In [17]:
def encode_query_clap(query: str) -> np.ndarray:
    """
    Encode a text query with the CLAP text encoder.
    Returns a normalised float32 vector of shape (CLAP_DIM,).
    The CLAP text and audio encoders share the same space:
    text queries can directly retrieve audio segments by cosine similarity.
    """
    if not CLAP_AVAILABLE:
        raise RuntimeError("CLAP model not available.")
    inputs = clap_processor(text=[query], return_tensors="pt", padding=True).to(clap_device)
    with torch.no_grad():
        vec = clap_model.get_text_features(**inputs)
        vec = clap_output_to_tensor(vec)
        vec = torch.nn.functional.normalize(vec, dim=-1).float()
    return vec.cpu().numpy()[0]


def clap_retrieve(query: str) -> List[AudioSegment]:
    """
    CLAP retrieval:
      1. Encode query with CLAP text encoder → query vector
      2. Cosine similarity search in QDRANT_CLAP_COLLECTION → audio segments
      3. Cross-encoder reranking on the *transcript* text of retrieved segments
         (CLAP retrieves by audio similarity; reranker refines by textual relevance)

    Note: BM25 is intentionally not used in this pipeline.
    CLAP embeddings already capture semantic audio content; keyword
    matching on the transcription would mix two retrieval signals
    that operate at different abstraction levels.
    """
    if not CLAP_AVAILABLE:
        print("[clap_retrieve] CLAP not available.")
        return []

    seen:        set                         = set()
    candidates:  List[Tuple[str, AudioSegment]] = []

    try:
        query_vec = encode_query_clap(query)
        result = client.query_points(
            collection_name = QDRANT_CLAP_COLLECTION,
            query           = query_vec.tolist(),
            limit           = RETRIEVER_K,
            with_payload    = True,
        )

        hits = result.points

        for h in hits:
            seg = clap_docstore.get(str(h.id))
            if seg is None or seg.content in seen:
                continue
            seen.add(seg.content)
            candidates.append((seg.transcript[:500], seg))
    except Exception as e:
        print(f"[clap_retrieve] Error: {e}")
        return []

    if not candidates:
        return []

    # Rerank on transcript text
    if ENABLE_RERANKING and RERANKER_AVAILABLE and reranker is not None:
        pairs  = [(query, t) for t, _ in candidates]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, [s for _, s in candidates]),
                        key=lambda x: x[0], reverse=True)
        final  = [s for _, s in ranked[:RERANKER_TOP_N]]
    else:
        final  = [s for _, s in candidates[:RERANKER_TOP_N]]

    print(f"[clap_retrieve] {len(hits)} hits → {len(final)} after rerank "
          f"(query: {query[:55]!r})")
    return final


# Sanity check
test_c = clap_retrieve("What is the attention mechanism?")
print(f"\nRetrieved {len(test_c)} segments:")
for i, s in enumerate(test_c):
    print(f"  [{i}] {s.timestamp_label}  {s.transcript[:80]!r}")


[clap_retrieve] 10 hits → 3 after rerank (query: 'What is the attention mechanism?')

Retrieved 3 segments:
  [0] [00:40 – 00:50]  'BERT is designed to pre-trained deep bi-directional representations from unlabel'
  [1] [03:28 – 03:38]  'Byrd alleviates the previously mentioned unidirectionality constraint by using a'
  [2] [06:08 – 06:18]  'During pre-training, the model is trained on unlabeled data over different pre-t'


---
## 5. Generation Model

Both pipelines pass **text only** to the LLM:
the Whisper pipeline passes transcribed chunk text,
the CLAP pipeline passes the Whisper transcript of the retrieved audio segment.


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GENERATOR_AVAILABLE     = False
USE_OLLAMA_GENERATOR    = False
gen_tokenizer           = None
gen_text_model          = None

if GENERATION_BACKEND == "ollama":
    try:
        from langchain_ollama import ChatOllama
        ollama_llm          = ChatOllama(model=GENERATION_MODEL, temperature=0)
        USE_OLLAMA_GENERATOR = True
        GENERATOR_AVAILABLE  = True
        print(f"✓ Ollama generator ready: {GENERATION_MODEL}")
    except Exception as e:
        print(f"✗ Ollama: {e}")
else:
    print(f"Loading HF generation model: {GENERATION_MODEL} ...")
    try:
        gen_tokenizer  = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
        gen_text_model = AutoModelForCausalLM.from_pretrained(
            GENERATION_MODEL,
            torch_dtype = HF_TORCH_DTYPE,
            device_map  = HF_DEVICE_MAP,
            trust_remote_code = True,
        )
        gen_text_model.eval()
        GENERATOR_AVAILABLE = True
        print(f"  ✓ Generator ready: {GENERATION_MODEL}")
    except Exception as e:
        print(f"  ✗ {e}")


def generate_answer(retrieved: List, question: str) -> str:
    """
    Build a text-only context from retrieved AudioChunk / AudioSegment objects
    and generate an answer with the local model.
    Always text-only: audio content is represented via its Whisper transcript.
    """
    if not GENERATOR_AVAILABLE:
        return "[Generator unavailable]"

    context_parts = []
    for item in retrieved:
        ts   = getattr(item, "timestamp_label", "")
        text = item.content.strip()
        if text:
            context_parts.append(f"{ts}\n{text}")

    context_str = "\n\n".join(context_parts) if context_parts else "[No context retrieved]"
    prompt = (
        "Answer the question using only the provided context from an audio transcript. "
        "If the context is insufficient, say so explicitly.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {question}"
    )

    if USE_OLLAMA_GENERATOR:
        return ollama_llm.invoke(prompt).content.strip()

    messages = [{"role": "user", "content": prompt}]
    if hasattr(gen_tokenizer, "apply_chat_template"):
        text_in = gen_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
    else:
        text_in = prompt

    inputs = gen_tokenizer([text_in], return_tensors="pt").to(gen_text_model.device)
    with torch.no_grad():
        generated = gen_text_model.generate(
            **inputs,
            max_new_tokens = GENERATION_MAX_NEW_TOKENS,
            do_sample      = False,
            temperature    = None,
            pad_token_id   = gen_tokenizer.eos_token_id,
        )
    new_tokens = generated[:, inputs.input_ids.shape[1]:]
    return gen_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


✓ Ollama generator ready: mistral-nemo:latest


In [19]:
# Quick generation test — both pipelines
q = "How is BERT positioned in regard to transformers?"

print("── Whisper pipeline ─────────────────────────────────────────────────")
w_docs = whisper_hybrid_retrieve(q)
w_answer = generate_answer(w_docs, q)
print(w_answer)

print("\n── CLAP pipeline ────────────────────────────────────────────────────")
c_docs = clap_retrieve(q)
c_answer = generate_answer(c_docs, q)
print(c_answer)


── Whisper pipeline ─────────────────────────────────────────────────
[whisper_retrieve] 11 candidates → 5 (query: 'How is BERT positioned in regard to transformers?')
BERT is positioned as an improvement over previous transformer models like OpenAI's GPT by incorporating bi-directional self-attention, allowing it to model context from both left and right sides of a token sequence. Unlike GPT, which uses constrained self-attention where each token can only attend to its left context, BERT can consider the entire sequence at once.

── CLAP pipeline ────────────────────────────────────────────────────
[clap_retrieve] 10 hits → 3 after rerank (query: 'How is BERT positioned in regard to transformers?')
BERT's transformer uses bi-directional self-attention, unlike other models like OpenAI GPT which use uni-directional attention.


---
## 6. Evaluation

Same TEST_SET and metrics as `rag_pipeline_clip.ipynb` for cross-notebook comparability.
An additional metric **timestamp_coverage** checks how many retrieved chunks
cover distinct, non-overlapping time ranges — a proxy for retrieval diversity.


In [20]:
import re, statistics
from typing import Dict, Any

# ── Shared TEST_SET (same questions as PDF experiments) ───────────────────────
TEST_SET = [
    {
        "question": "What is scaled dot-product attention?",
        "context_keywords": ["softmax", "dot product", "queries", "keys", "values"],
        "answer_keywords":  ["softmax", "scale", "queries", "keys"],
    },
    {
        "question": "What is multi-head attention?",
        "context_keywords": ["multi-head", "projection", "parallel", "heads"],
        "answer_keywords":  ["head", "parallel", "projection"],
    },
    {
        "question": "What optimizer was used to train the Transformer?",
        "context_keywords": ["adam", "optimizer", "warmup", "learning rate"],
        "answer_keywords":  ["adam", "warmup"],
    },
    {
        "question": "What is the role of positional encoding?",
        "context_keywords": ["positional", "encoding", "position", "sequence"],
        "answer_keywords":  ["position", "order", "encoding"],
    },
    {
        "question": "How does the encoder differ from the decoder?",
        "context_keywords": ["encoder", "decoder", "masked", "self-attention"],
        "answer_keywords":  ["encoder", "decoder", "masked"],
    },
]

STOPWORDS = {"the","a","an","is","in","of","to","and","or","it","that","this",
             "with","for","on","are","was","be","by","at","as","has","have","had","from"}

def tokenize(text: str) -> set:
    return set(re.findall(r"[a-z0-9]+", text.lower()))

def context_recall(docs, keywords: List[str]) -> float:
    ctx = " ".join(d.content for d in docs).lower()
    return sum(1 for kw in keywords if kw.lower() in ctx) / max(len(keywords), 1)

def answer_faithfulness(answer: str, docs) -> float:
    ctx_tok  = tokenize(" ".join(d.content for d in docs)) - STOPWORDS
    sents    = [s.strip() for s in re.split(r"[.!?]", answer) if len(s.strip()) > 10]
    if not sents: return 1.0
    return sum(1 for s in sents if (tokenize(s) - STOPWORDS) & ctx_tok) / len(sents)

def answer_relevance(answer: str, question: str) -> float:
    q_tok = tokenize(question) - STOPWORDS
    a_tok = tokenize(answer)   - STOPWORDS
    return len(q_tok & a_tok) / max(len(q_tok), 1)

def retrieval_precision(docs, answer_keywords: List[str]) -> float:
    if not docs or not answer_keywords: return 0.0
    return sum(1 for d in docs if any(kw.lower() in d.content.lower()
                                      for kw in answer_keywords)) / len(docs)

def timestamp_coverage(docs) -> float:
    """
    Fraction of total audio covered by retrieved segments, capped at 1.0.
    Measures retrieval diversity — higher = more distinct time ranges covered.
    """
    if not docs or AUDIO_DURATION_SECS <= 0:
        return 0.0
    # Merge overlapping intervals
    intervals = sorted([(getattr(d, "start_sec", 0), getattr(d, "end_sec", 0))
                        for d in docs], key=lambda x: x[0])
    merged, total_covered = [], 0.0
    for s, e in intervals:
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    total_covered = sum(e - s for s, e in merged)
    return min(total_covered / AUDIO_DURATION_SECS, 1.0)

METRICS = ["context_recall", "answer_faithfulness", "answer_relevance",
           "retrieval_precision", "timestamp_coverage"]

print(f"Evaluation ready — {len(TEST_SET)} questions, {len(METRICS)} metrics.")


Evaluation ready — 5 questions, 5 metrics.


In [21]:
def run_audio_evaluation(
    retrieve_fn,
    test_set: List[Dict[str, Any]],
    label: str,
) -> List[Dict]:
    """Generic evaluation loop for any audio retrieval function."""
    results = []
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"{'='*60}")

    for item in test_set:
        q    = item["question"]
        c_kw = item["context_keywords"]
        a_kw = item["answer_keywords"]

        try:
            retrieved = retrieve_fn(q)
        except Exception as e:
            print(f"  ✗ Retrieval: {q!r} — {e}")
            results.append({"question": q, "pipeline": label, "error": str(e)})
            continue
        try:
            answer = generate_answer(retrieved, q)
        except Exception as e:
            print(f"  ✗ Generation: {q!r} — {e}")
            results.append({"question": q, "pipeline": label, "error": str(e)})
            continue

        result = {
            "question":             q,
            "pipeline":             label,
            "context_recall":       round(context_recall(retrieved, c_kw), 3),
            "answer_faithfulness":  round(answer_faithfulness(answer, retrieved), 3),
            "answer_relevance":     round(answer_relevance(answer, q), 3),
            "retrieval_precision":  round(retrieval_precision(retrieved, a_kw), 3),
            "timestamp_coverage":   round(timestamp_coverage(retrieved), 3),
            "n_retrieved":          len(retrieved),
            "answer_preview":       answer[:150],
        }
        results.append(result)
        print(f"  Q: {q[:60]}")
        print(f"     recall={result['context_recall']:.2f}  "
              f"faith={result['answer_faithfulness']:.2f}  "
              f"rel={result['answer_relevance']:.2f}  "
              f"prec={result['retrieval_precision']:.2f}  "
              f"ts_cov={result['timestamp_coverage']:.3f}")
    return results


# Run both pipelines
whisper_eval_results = run_audio_evaluation(whisper_hybrid_retrieve, TEST_SET, "Whisper")
clap_eval_results = run_audio_evaluation(clap_retrieve, TEST_SET, "CLAP")



Evaluating: Whisper
[whisper_retrieve] 13 candidates → 5 (query: 'What is scaled dot-product attention?')
  Q: What is scaled dot-product attention?
     recall=0.00  faith=1.00  rel=0.80  prec=0.00  ts_cov=0.218
[whisper_retrieve] 13 candidates → 5 (query: 'What is multi-head attention?')
  Q: What is multi-head attention?
     recall=0.00  faith=1.00  rel=0.75  prec=0.00  ts_cov=0.221
[whisper_retrieve] 11 candidates → 5 (query: 'What optimizer was used to train the Transformer?')
  Q: What optimizer was used to train the Transformer?
     recall=0.00  faith=1.00  rel=0.80  prec=0.00  ts_cov=0.220
[whisper_retrieve] 13 candidates → 5 (query: 'What is the role of positional encoding?')
  Q: What is the role of positional encoding?
     recall=0.25  faith=0.50  rel=0.50  prec=0.00  ts_cov=0.223
[whisper_retrieve] 12 candidates → 5 (query: 'How does the encoder differ from the decoder?')
  Q: How does the encoder differ from the decoder?
     recall=0.50  faith=1.00  rel=0.80  prec=0.4

---
## 7. Comparison: Whisper vs CLAP


In [22]:
def aggregate_results(results: List[Dict], metrics: List[str]) -> Dict:
    valid = [r for r in results if "error" not in r]
    if not valid: return {}
    return {m: round(statistics.mean(r[m] for r in valid), 3) for m in metrics}

w_agg = aggregate_results(whisper_eval_results, METRICS)
c_agg = aggregate_results(clap_eval_results,    METRICS)

print("=" * 75)
print("AUDIO PIPELINE COMPARISON")
print(f"  Whisper : {WHISPER_MODEL}")
print(f"  CLAP    : {CLAP_MODEL}")
print(f"  LLM     : {GENERATION_MODEL}")
print(f"  Audio   : {AUDIO_PATH}")
print("=" * 75)
print(f"{'Metric':<26} {'Whisper':>9} {'CLAP':>9} {'Delta':>10}  Winner")
print("-" * 75)
for m in METRICS:
    w = w_agg.get(m, float("nan"))
    c = c_agg.get(m, float("nan"))
    d = c - w
    winner = "Whisper" if w > c else ("CLAP" if c > w else "tie")
    print(f"{m:<26} {w:>9.3f} {c:>9.3f} {'+' if d>=0 else ''}{d:>9.3f}  {winner}")
print("-" * 75)

# Per-question breakdown
print("\nPer-question breakdown:")
hdr = f"{'Question':<43} {'Pipe':<9} {'recall':>7} {'faith':>7} {'rel':>6} {'prec':>6} {'ts_cov':>8}"
print(hdr)
print("-" * len(hdr))
for wr, cr in zip(whisper_eval_results, clap_eval_results):
    q = wr["question"][:41]
    for r, tag in [(wr, "Whisper"), (cr, "CLAP")]:
        if "error" not in r:
            print(f"{q if tag=='Whisper' else '':<43} {tag:<9} "
                  f"{r['context_recall']:>7.3f} {r['answer_faithfulness']:>7.3f} "
                  f"{r['answer_relevance']:>6.3f} {r['retrieval_precision']:>6.3f} "
                  f"{r['timestamp_coverage']:>8.3f}")
    print()


AUDIO PIPELINE COMPARISON
  Whisper : deepdml/faster-whisper-large-v3-turbo-ct2
  CLAP    : laion/larger_clap_general
  LLM     : mistral-nemo:latest
  Audio   : ./content/audio.mp3
Metric                       Whisper      CLAP      Delta  Winner
---------------------------------------------------------------------------
context_recall                 0.150     0.150 +    0.000  tie
answer_faithfulness            0.900     0.900 +    0.000  tie
answer_relevance               0.730     0.730 +    0.000  tie
retrieval_precision            0.080     0.133 +    0.053  CLAP
timestamp_coverage             0.221     0.048    -0.173  Whisper
---------------------------------------------------------------------------

Per-question breakdown:
Question                                    Pipe       recall   faith    rel   prec   ts_cov
--------------------------------------------------------------------------------------------
What is scaled dot-product attention?       Whisper     0.000   1.000 

---
## 8. Extended Evaluation Grid

Evaluates all combinations of:

| Dimension | Values |
|---|---|
| `pipeline` | `whisper` · `clap` |
| `retrieval_mode` | `dense_only` · `bm25_only` (Whisper only) · `hybrid` · `hybrid_rerank` |
| `whisper_model` | recorded as metadata |
| `clap_model` | recorded as metadata |
| `llm_model` | recorded as metadata |

> To test a different Whisper/CLAP/LLM model: change the config vars,
> re-run the relevant loading + indexing cells, then call `run_extended_audio_evaluation()`
> with a new label to accumulate results for comparison.


In [23]:
def make_audio_retrieval_fn(pipeline: str, retrieval_mode: str):
    """
    Factory for audio retrieval functions.
    pipeline       : 'whisper' | 'clap'
    retrieval_mode : 'dense_only' | 'bm25_only' | 'hybrid' | 'hybrid_rerank'

    'bm25_only' is only meaningful for the Whisper pipeline.
    For CLAP, 'bm25_only' falls back to dense (logged as warning).
    """
    use_dense  = retrieval_mode in ("dense_only",  "hybrid", "hybrid_rerank")
    use_bm25_  = retrieval_mode in ("bm25_only",   "hybrid", "hybrid_rerank")
    use_rerank = (retrieval_mode == "hybrid_rerank")

    def _retrieve(query: str) -> List:
        seen:       set                  = set()
        candidates: List[Tuple[str, Any]] = []

        if pipeline == "clap":
            if use_bm25_ and not use_dense:
                print(f"  [warn] bm25_only not applicable to CLAP — using dense instead.")
            # CLAP: always dense (audio embedding space)
            try:
                qvec = encode_query_clap(query)
                hits = client.search(
                    collection_name = QDRANT_CLAP_COLLECTION,
                    query_vector    = qvec.tolist(),
                    limit           = RETRIEVER_K,
                    with_payload    = True,
                )
                for h in hits:
                    seg = clap_docstore.get(str(h.id))
                    if seg and seg.content not in seen:
                        seen.add(seg.content)
                        candidates.append((seg.transcript[:500], seg))
            except Exception as e:
                print(f"  [clap dense] {e}")

        else:  # whisper
            if use_dense:
                try:
                    for lc in whisper_vector_store.similarity_search(query, k=RETRIEVER_K):
                        uid   = lc.metadata.get("doc_id")
                        chunk = whisper_docstore.get(uid)
                        if chunk is None:
                            chunk = AudioChunk(content=lc.page_content,
                                               start_sec=lc.metadata.get("start_sec",0),
                                               end_sec=lc.metadata.get("end_sec",0))
                        if chunk.content not in seen:
                            seen.add(chunk.content)
                            candidates.append((lc.page_content, chunk))
                except Exception as e:
                    print(f"  [whisper dense] {e}")
            if use_bm25_:
                for score, chunk in whisper_bm25.retrieve(query, top_k=RETRIEVER_K):
                    if chunk.content not in seen:
                        seen.add(chunk.content)
                        candidates.append((chunk.content[:500], chunk))

        if not candidates:
            return []

        if use_rerank and RERANKER_AVAILABLE and reranker is not None:
            pairs  = [(query, t) for t, _ in candidates]
            scores = reranker.predict(pairs)
            ranked = sorted(zip(scores, [d for _, d in candidates]),
                            key=lambda x: x[0], reverse=True)
            return [d for _, d in ranked[:RERANKER_TOP_N]]
        return [d for _, d in candidates[:RERANKER_TOP_N]]

    return _retrieve


def run_extended_audio_evaluation(
    test_set:          List[Dict[str, Any]],
    pipelines:         tuple = ("whisper", "clap"),
    retrieval_modes:   tuple = ("dense_only", "bm25_only", "hybrid", "hybrid_rerank"),
    whisper_label:     str   = WHISPER_MODEL,
    clap_label:        str   = CLAP_MODEL,
    llm_label:         str   = GENERATION_MODEL,
) -> List[Dict]:
    """Run the full (pipeline × retrieval_mode) evaluation grid."""
    from itertools import product as iproduct
    combos      = list(iproduct(pipelines, retrieval_modes))
    all_results = []
    print(f"Extended audio evaluation: {len(combos)} combos × {len(test_set)} questions")

    for pipe, ret_mode in combos:
        label       = f"{pipe}+{ret_mode}"
        retrieve_fn = make_audio_retrieval_fn(pipe, ret_mode)
        print(f"\n── {label} {'─'*max(0,50-len(label))}")

        for item in test_set:
            q    = item["question"]
            c_kw = item["context_keywords"]
            a_kw = item["answer_keywords"]
            base = {"pipeline": pipe, "retrieval_mode": ret_mode,
                    "whisper_model": whisper_label, "clap_model": clap_label,
                    "llm_model": llm_label, "question": q}
            try:
                retrieved = retrieve_fn(q)
            except Exception as e:
                all_results.append({**base, "error": str(e)}); continue
            try:
                answer = generate_answer(retrieved, q)
            except Exception as e:
                all_results.append({**base, "error": str(e)}); continue

            all_results.append({
                **base,
                "context_recall":      round(context_recall(retrieved, c_kw), 3),
                "answer_faithfulness": round(answer_faithfulness(answer, retrieved), 3),
                "answer_relevance":    round(answer_relevance(answer, q), 3),
                "retrieval_precision": round(retrieval_precision(retrieved, a_kw), 3),
                "timestamp_coverage":  round(timestamp_coverage(retrieved), 3),
                "n_retrieved":         len(retrieved),
                "answer_preview":      answer[:120],
            })

        combo_ok = [r for r in all_results
                    if r["pipeline"]==pipe and r["retrieval_mode"]==ret_mode
                    and "error" not in r]
        if combo_ok:
            for m in METRICS:
                print(f"  {m:<26}: {statistics.mean(r[m] for r in combo_ok):.3f}")

    return all_results


In [24]:
extended_audio_results = run_extended_audio_evaluation(TEST_SET)


Extended audio evaluation: 8 combos × 5 questions

── whisper+dense_only ────────────────────────────────
  context_recall            : 0.100
  answer_faithfulness       : 0.700
  answer_relevance          : 0.690
  retrieval_precision       : 0.133
  timestamp_coverage        : 0.130

── whisper+bm25_only ─────────────────────────────────
  context_recall            : 0.150
  answer_faithfulness       : 1.000
  answer_relevance          : 0.730
  retrieval_precision       : 0.120
  timestamp_coverage        : 0.218

── whisper+hybrid ────────────────────────────────────
  context_recall            : 0.200
  answer_faithfulness       : 0.900
  answer_relevance          : 0.690
  retrieval_precision       : 0.120
  timestamp_coverage        : 0.217

── whisper+hybrid_rerank ─────────────────────────────
  context_recall            : 0.150
  answer_faithfulness       : 0.900
  answer_relevance          : 0.730
  retrieval_precision       : 0.080
  timestamp_coverage        : 0.221

── cl

In [25]:
# ── Extended results summary ──────────────────────────────────────────────────
from collections import defaultdict

grouped: dict = defaultdict(list)
for r in extended_audio_results:
    if "error" not in r:
        grouped[(r["pipeline"], r["retrieval_mode"])].append(r)

print("\n" + "=" * 90)
print("EXTENDED AUDIO EVALUATION SUMMARY")
print(f"  Whisper: {WHISPER_MODEL}  |  CLAP: {CLAP_MODEL}  |  LLM: {GENERATION_MODEL}")
print("=" * 90)
print(f"{'Pipeline':<10} {'Retrieval':<16} {'recall':>8} {'faith':>7} {'rel':>6} {'prec':>6} {'ts_cov':>8}")
print("-" * 90)
for key in sorted(grouped.keys()):
    rows = grouped[key]
    avgs = {m: round(statistics.mean(r[m] for r in rows), 3) for m in METRICS}
    print(f"{key[0]:<10} {key[1]:<16} "
          f"{avgs['context_recall']:>8.3f} {avgs['answer_faithfulness']:>7.3f} "
          f"{avgs['answer_relevance']:>6.3f} {avgs['retrieval_precision']:>6.3f} "
          f"{avgs['timestamp_coverage']:>8.3f}")
print("-" * 90)

print("\nBest configuration per metric:")
for m in METRICS:
    if not grouped: continue
    best = max(grouped.keys(), key=lambda k: statistics.mean(r[m] for r in grouped[k]))
    val  = statistics.mean(r[m] for r in grouped[best])
    print(f"  {m:<26}: {best[0]}+{best[1]:<22} ({val:.3f})")

# ── Save results for cross-notebook comparison ────────────────────────────────
import json, pathlib
out_path = pathlib.Path(PERSIST_DIR) / "audio_eval_results.json"
with open(out_path, "w") as f:
    json.dump(extended_audio_results, f, ensure_ascii=False, indent=2)
print(f"\nResults saved → {out_path}")
print("Load in comparison notebook with:")
print(f"  import json")
print(f"  audio_results = json.load(open('{out_path}'))")



EXTENDED AUDIO EVALUATION SUMMARY
  Whisper: deepdml/faster-whisper-large-v3-turbo-ct2  |  CLAP: laion/larger_clap_general  |  LLM: mistral-nemo:latest
Pipeline   Retrieval          recall   faith    rel   prec   ts_cov
------------------------------------------------------------------------------------------
clap       bm25_only           0.000   0.000  0.390  0.000    0.000
clap       dense_only          0.000   0.000  0.390  0.000    0.000
clap       hybrid              0.000   0.000  0.390  0.000    0.000
clap       hybrid_rerank       0.000   0.000  0.390  0.000    0.000
whisper    bm25_only           0.150   1.000  0.730  0.120    0.218
whisper    dense_only          0.100   0.700  0.690  0.133    0.130
whisper    hybrid              0.200   0.900  0.690  0.120    0.217
whisper    hybrid_rerank       0.150   0.900  0.730  0.080    0.221
------------------------------------------------------------------------------------------

Best configuration per metric:
  context_recall     